In [1]:
import numpy as np, pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC, SVR
from sklearn.metrics import r2_score, classification_report



# morgan packages
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

## manipulating the model data

In [2]:
df= pd.read_csv("df_ic50.csv")
print(df.shape)
df.head()

(2952, 46)


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,NaN,NaN,105376,[],CHEMBL660784,Inhibitory concentration against cyclin-depend...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,36.00
1,NaN,NaN,106512,[],CHEMBL661128,Inhibitory concentration against cyclin-depend...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,2.00
2,NaN,NaN,107704,[],CHEMBL661128,Inhibitory concentration against cyclin-depend...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,1000.00
3,NaN,NaN,107706,[],CHEMBL661128,Inhibitory concentration against cyclin-depend...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,1000.00
4,NaN,NaN,108039,[],CHEMBL661130,Inhibition of Cyclin-dependent kinase 2,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,0.03


In [3]:
df.standard_value.isna().sum()

39

In [4]:
#dropping the na values in standard value column
df.dropna(subset=["standard_value"], axis=0, inplace=True)

In [5]:
df.standard_value.isna().sum()

0

In [6]:
df.standard_units.value_counts()

standard_units
nM         2861
ug.mL-1      52
Name: count, dtype: int64

In [7]:
# excluding the ug.mL-1  readings
# df['standard_units2']=np.where (df.standard_type=="ug.mL-1", "NA", "nM")
df=df[df["standard_units"]!= "ug.mL-1"]
df.standard_units.value_counts()

standard_units
nM    2861
Name: count, dtype: int64

In [8]:
#selectiong the 5 nM as a cutoff point
df["activity"]=np.where(df["standard_value"]<=10, "active", "inactive")
df.activity.value_counts()

activity
inactive    2468
active       393
Name: count, dtype: int64

In [9]:
type(df.canonical_smiles)

pandas.core.series.Series

In [10]:
def create_molecule(smiles_string):
  try:
    return Chem.MolFromSmiles(smiles_string)
  except:
    return None  # Handle cases where SMILES string is invalid

# Apply the function with error handling
df["Molecule"] = df["canonical_smiles"].apply(create_molecule)
df = df[df['Molecule'].notnull()]  # Keep rows with valid molecules


In [11]:
# df["Molecule"]=df["canonical_smiles"].apply(Chem.MolFromSmiles)
df["Fingerprint"]=df["Molecule"].apply(lambda x:AllChem.GetMorganFingerprintAsBitVect(x,2))

In [12]:
df.Fingerprint


0       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...
3       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...
4       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                              ...                        
2947    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
2948    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2949    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
2950    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
2951    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
Name: Fingerprint, Length: 2857, dtype: object

## designing the model

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    list(df['Fingerprint']),
    df['activity'],
    test_size=0.3,
    random_state=42
)

In [14]:
# # Training the models
# models={
#     "lr":LogisticRegression(C=1),
#     "svc":SVC(),
#     "DTC":DecisionTreeClassifier(),
#     "RFC":RandomForestClassifier(n_estimators=1000, max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
#     "GBC":GradientBoostingClassifier()
# }


# for names, model in models.items():
#     model.fit(X_train, y_train)
#     Y_pred= model.predict(X_test)
#     print(classification_report(Y_pred, y_test)+names)

In [15]:
# considering the RFC as the best model
model=RandomForestClassifier(max_depth=30, min_samples_split=2, n_estimators=61, min_samples_leaf=1, random_state=123)
model.fit(X_train, y_train)

y_predict=model.predict(X_test)
print(classification_report(y_predict, y_test))

              precision    recall  f1-score   support

      active       0.70      0.79      0.74        99
    inactive       0.97      0.96      0.96       759

    accuracy                           0.94       858
   macro avg       0.84      0.87      0.85       858
weighted avg       0.94      0.94      0.94       858



## preparing all of the pbuchem compounds using our model

In [16]:
df_pubchem= pd.read_csv("all_active_3cols.csv")
df_pubchem.shape

(5206, 7)

In [26]:
df_pubchem=df_pubchem.drop_duplicates(subset=" cid")
df_pubchem.shape

(3106, 7)

In [ ]:
df_active=pd.read_csv("")

In [27]:
df_pubchem.columns

Index(['Unnamed: 0', 'canonicalsmiles', 'Molecule', 'Fingerprint', 'predicted',
       ' cid', 'cmpdname'],
      dtype='object')

In [28]:
df_pubchem.canonicalsmiles.isna().sum()

0

In [19]:
# #creating the Mol from smiles
def create_molecule(smiles_string):
  try:
    return Chem.MolFromSmiles(smiles_string)
  except:
    return None  # Handle cases where SMILES string is invalid

# Apply the function with error handling
df_pubchem["Molecule"] = df_pubchem["canonicalsmiles"].apply(create_molecule)

[20:53:33] WARNING: not removing hydrogen atom without neighbors
[20:53:34] WARNING: not removing hydrogen atom without neighbors
[20:53:34] WARNING: not removing hydrogen atom without neighbors
[20:53:34] WARNING: not removing hydrogen atom without neighbors
[20:53:34] WARNING: not removing hydrogen atom without neighbors


In [20]:
df_pubchem = df_pubchem[df_pubchem['Molecule'].notnull()]  # Keep rows with valid molecules


In [21]:
# #creating the morgan fingerprint column
df_pubchem["Fingerprint"]=df_pubchem["Molecule"].apply(lambda x:AllChem.GetMorganFingerprintAsBitVect(x,2))


In [47]:
#predicting the activity
df_pubchem["predicted"]=model.predict(list(df_pubchem["Fingerprint"]))
#extracting the active probability
active_class = 0

# Get predicted probabilities
predicted_proba = model.predict_proba(list(df_pubchem["Fingerprint"]))

# Extract probabilities for the active class
df_pubchem["predicted_proba"] = predicted_proba[:, active_class]

In [49]:
print(df_pubchem.predicted_proba.value_counts( ascending=True))# all of them are active
df_pubchem.head()

predicted_proba
0.651991     1
0.555001     1
0.518553     1
0.574146     1
0.590772     1
            ..
0.836066    34
0.529169    39
0.508237    47
0.512776    50
0.868852    71
Name: count, Length: 1239, dtype: int64


,Unnamed: 0,canonicalsmiles,Molecule,Fingerprint,predicted,cid,cmpdname,predicted_proba
0,0,CC12C(C(CC(O1)N3C4=C(C=C(C=C4)O)C5=C6C(=C7C8=C...,<rdkit.Chem.rdchem.Mol object at 0x7f8dc7f757e0>,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",active,1605,"11,25-Dihydroxy-3-methoxy-2-methyl-4-(methylam...",0.645129
1,1,CC12C(C(CC(O1)N3C4=C(C=C(C=C4)O)C5=C6C(=C7C8=C...,<rdkit.Chem.rdchem.Mol object at 0x7f8dc7f758c0>,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",active,45027788,"3,11-Dihydroxystaurosporine",0.645129
4,4,C1CCC(C1)N2C=NC3=C(N=C(N=C32)NC4CCC(CC4)N)NCC5...,<rdkit.Chem.rdchem.Mol object at 0x7f8dc7f75a10>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",active,3543,2-[trans-(4-Aminocyclohexyl)amino]-6-(benzyl-a...,0.711335
5,5,C1CCC(CC1)COC2=NC(=NC3=C2NC=N3)NC4=CC=C(C=C4)S...,<rdkit.Chem.rdchem.Mol object at 0x7f8dc7f75a80>,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",active,4566,O6-Cyclohexylmethoxy-2-(4'-sulphamoylanilino) ...,0.604265
6,6,CC12C(C(CC(O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=...,<rdkit.Chem.rdchem.Mol object at 0x7f8dc7f75700>,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",active,5279,Staurosporine HCl,0.868852


In [50]:
df_pubchem.to_csv("predicted3106with_probs.csv")